# DistilBERT Colab pilot and training adapter

Use a GPU runtime. This notebook contains orchestration only: governed data and model logic remains in the repository package. Before running it, manually commit and push the repository, and upload `training_dataset.parquet` to the private Drive path below. The notebook never commits or pushes changes.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/MurrayMint7/AIDI_artefact.git'
REPO_ROOT = '/content/AIDI_artefact'
PRIVATE_ROOT = '/content/drive/MyDrive/AIDI_artefact/private'
DATASET_PATH = f'{PRIVATE_ROOT}/training_dataset.parquet'
MODEL_CACHE = f'{PRIVATE_ROOT}/huggingface-cache'
EVIDENCE_DIR = f'{PRIVATE_ROOT}/evidence'

In [ ]:
from pathlib import Path
if not Path(DATASET_PATH).is_file():
    raise FileNotFoundError(f'Upload the governed Parquet handoff to {DATASET_PATH}')
if Path(REPO_ROOT).exists():
    raise RuntimeError(f'{REPO_ROOT} already exists; restart for a clean clone')
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)
subprocess.run(['git', '-C', REPO_ROOT, 'rev-parse', 'HEAD'], check=True)

In [ ]:
%cd /content/AIDI_artefact
%pip install -q -r requirements-transformer.txt
import importlib.metadata
import sys
for package in ['torch', 'transformers', 'datasets', 'accelerate', 'fsspec']:
    print(package, importlib.metadata.version(package))
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

In [ ]:
command = [
    sys.executable, '-m', 'amazon_sentiment', 'throughput',
    '--config', 'config/distilbert.yaml',
    '--dataset', DATASET_PATH,
    '--token-summary', 'artifacts/metrics/distilbert_token_length_summary.json',
    '--output-dir', 'artifacts/metrics',
    '--cache-dir', MODEL_CACHE,
]
subprocess.run(command, check=True)

In [ ]:
import json
import shutil
evidence = Path(EVIDENCE_DIR)
evidence.mkdir(parents=True, exist_ok=True)
for name in [
    'distilbert_throughput_results.json',
    'distilbert_pilot_decision.json',
    'distilbert_colab_environment.json',
]:
    shutil.copy2(Path('artifacts/metrics') / name, evidence / name)
print(json.loads(Path('artifacts/metrics/distilbert_pilot_decision.json').read_text()))